# Voronoi Partitioning and the Inverted-File (IVF) Index — companion notebook

The **narrative** half of the Python pillar. The tested implementation lives next door in
[`ivf_voronoi_partitioning.py`](ivf_voronoi_partitioning.py); here we import it and walk the topic
section by section, so every claim renders as executed output.

The inverted-file index is the non-exhaustive half of vector search: a **coarse quantizer** (Lloyd's
k-means) partitions the space into Voronoi cells, each vector is filed under its nearest centroid,
and a query is compared only against the few cells nearest to it. **IVFADC** then product-quantizes
the *residual* after the coarse partition. So this module imports the k-means core and the product
quantizer from the previous topics and never reimplements them; `ivf_voronoi_partitioning.py` owns
the numbers, its `test_*` assertions encode the claims, and `viz_constants()` prints what the
laboratory mirrors.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "ivf-voronoi-partitioning",
              pathlib.Path("notebooks/ivf-voronoi-partitioning")):
    if (_cand / "ivf_voronoi_partitioning.py").exists():
        sys.path.insert(0, str(_cand)); break
import numpy as np
import ivf_voronoi_partitioning as ivf
from vector_quantization_lloyd_max import finance_dataset
X, _, queries = finance_dataset()
C = ivf.coarse_quantizer(X, ivf.NLIST, seed=0)
labels, lists = ivf.inverted_lists(X, C)

## Movement 1 — the partition (IVF)

### 1. Exhaustive probe is exact; a single probe scans a sliver

Probing all `nlist` cells with the exact distance *is* brute-force nearest-neighbor search, so its
recall is 1. Probing a single cell scans only a small fraction of the database — the candidate-set
reduction that buys the speedup.

In [ ]:
ivf.test_full_probe_is_exact()
ivf.test_candidate_reduction()

### 2. The recall / scan frontier

Sweeping `nprobe` traces the trade the index lives along: recall climbs toward 1 as the fraction of
the database scanned grows. Recall is monotone in `nprobe` — probing more cells never loses a neighbor.

In [ ]:
for r in ivf.recall_vs_nprobe(queries, X, C, lists, ivf.NPROBE_GRID):
    print(f"  nprobe={r['nprobe']:>3}: recall={r['recall']:.3f}  fraction scanned={r['frac']:.3f}")
ivf.test_recall_monotone_in_nprobe()

### 3. The boundary effect (the honest catch)

A query's true nearest neighbor can sit in a cell whose centroid is **not** the query's nearest
centroid — a neighbor across a Voronoi boundary. So recall at `nprobe = 1` is strictly below 1, and
multi-probe (`nprobe > 1`) recovers it. This is the approximation the index makes.

In [ ]:
ivf.test_boundary_effect()

## Movement 2 — the residual (IVFADC)

### 4. The coarse quantizer removes between-cell variance

Subtracting each vector's cell centroid leaves a residual with strictly smaller total variance:
the coarse partition has already captured the between-cell structure. Product-quantizing the
*residual* therefore spends the same bits on a smaller-variance signal.

In [ ]:
raw, res = ivf.variance_reduction(X, C, labels)
print(f"  total variance: {raw:.2f} (raw) -> {res:.2f} (residual), {100*(1-res/raw):.0f}% removed")
ivf.test_residual_variance_reduction()

### 5. IVFADC beats flat PQ at equal bits

At the same product-quantization budget, encoding the residual (IVFADC) reaches at least the recall
of encoding the raw vector (flat PQ), because the residual is cheaper to quantize. Direction, not a
magic number.

In [ ]:
ivf.test_ivfadc_beats_flat_pq_equal_bits()

## Cross-check and viz constants

The IVF plumbing is verified against a brute-force top-k, and `viz_constants()` prints the toy
Voronoi cloud and boundary query, the recall/scan frontier, and the IVFADC residual numbers — the
exact values the laboratory mirrors to the decimal.

In [ ]:
ivf.validate_against_bruteforce()
ivf.viz_constants()

## All claims verified

Every theorem and the honest caveat are executed assertions above. The recall frontier, the boundary
effect, the residual-variance reduction, and the IVFADC comparison are the exact numbers the
laboratory mirrors. Re-run top to bottom to reproduce the topic.